# Anomaly and outlier detection

Finding the rare, the unexpected, the compromised -- in user behaviour, network traffic, transactions,
sensors, or medical signals -- is anomaly detection, and it is harder than it looks because anomalies are by
definition scarce and usually unlabelled. We work through it, grounded in real user-process
activity logs (which processes each user ran, and when). We build the full toolkit from first principles:
robust univariate outlier theory and why naive z-scores fail, the multivariate Gaussian and Mahalanobis
distance, density estimation with Gaussian mixtures and Dirichlet-process mixtures, order-aware sequence
models (Markov chains and mixtures of sequences), temporal and rate anomalies extracted from the timestamps,
extreme-value theory for principled thresholds, conformal anomaly detection with guaranteed false-alarm rates
and false-discovery control, and a disciplined evaluation by injected attacks of known severity. The work is
modelling; plotting lives in one helper.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, io
import matplotlib.pyplot as plt
from scipy.stats import chi2, t as tdist, genpareto
from mixle.stats import (CategoricalEstimator, SequenceEstimator, MixtureEstimator, MarkovChainEstimator,
                        MultivariateGaussianEstimator, PoissonEstimator, ExponentialEstimator,
                        DirichletProcessMixtureEstimator, GammaDistribution, seq_encode)
from mixle.inference.estimation import optimize
rng = np.random.RandomState(0)
def panel(ncol=1, w=5.0, h=3.0):
    fig, ax = plt.subplots(1, ncol, figsize=(w * ncol, h)); return fig, (ax if ncol > 1 else [ax])
opt = lambda data, est, its=60, seed=1: optimize(data, est, max_its=its, rng=np.random.RandomState(seed), out=io.StringIO())
load = lambda split: pd.read_csv('../../data/user_proc/user_proc_data_%s.csv' % split)
train_df, valid_df, test_df = load('train'), load('valid'), load('test')
PROCS = sorted(train_df['process_name'].unique()); PIDX = {p: i for i, p in enumerate(PROCS)}
def by_user(d):
    out = {}
    for u, g in d.sort_values('time').groupby('user_name'):
        out[u] = (list(g['process_name']), np.array(g['time'], float))
    return out
train_u, valid_u, test_u = by_user(train_df), by_user(valid_df), by_user(test_df)
print('%d processes; train %d users / %d events, valid %d users, test %d users'
      % (len(PROCS), len(train_u), len(train_df), len(valid_u), len(test_u)))

50 processes; train 300 users / 14263 events, valid 300 users, test 300 users


## 1. What an anomaly is, and why detection is hard

An anomaly is an observation generated by a different process than the bulk of the data -- formally, a point
in a low-density region of the normal distribution. Three structural difficulties shape every method. First,
anomalies are rare, so we almost never have enough labels to train a classifier; detection is fundamentally
unsupervised, learning the normal and flagging deviations. Second, "deviation" must be measured against a
model of normal, and the model choice (distance, density, sequence, rate) determines what kinds of anomaly are
even visible. Third, evaluation without labels requires injecting known anomalies of controlled severity. In
this data a compromised account shows up as one of three signatures: a shifted process mix (masquerade), new
or rare processes, or anomalous timing and rate. We will build a detector for each and combine them.

In [2]:
# the normal fingerprint of a user: how concentrated and how repetitive their process usage is
def proc_hist(procs):
    h = np.zeros(len(PROCS));
    for p in procs: h[PIDX[p]] += 1
    return h / h.sum()
H = np.array([proc_hist(train_u[u][0]) for u in train_u])
ent = -np.sum(np.where(H > 0, H * np.log(H + 1e-12), 0), 1)
print('typical user runs %.0f events across %.0f distinct processes; process-usage entropy %.2f +- %.2f nats'
      % (np.mean([len(train_u[u][0]) for u in train_u]), np.mean([len(set(train_u[u][0])) for u in train_u]), ent.mean(), ent.std()))
print('anomalies will be points that violate this concentration, the process mix, the order, or the timing')

typical user runs 48 events across 11 distinct processes; process-usage entropy 1.96 +- 0.38 nats
anomalies will be points that violate this concentration, the process mix, the order, or the timing


## 2. Univariate outlier theory: why the z-score fails

The oldest detector is the z-score: flag points more than three standard deviations from the mean. It has a
fatal flaw -- the mean and standard deviation are themselves corrupted by the outliers they are meant to find
(low breakdown point). One gross outlier inflates the standard deviation so much that it masks itself
(masking), and can make a clean point look extreme (swamping). The Grubbs / generalised-ESD test adds a
calibrated critical value for "is the most extreme point an outlier" under normality, but it is built on the
same non-robust mean and standard deviation, so a cluster of outliers masks even it. The genuinely robust
replacement uses the median and the median absolute deviation (MAD), which tolerate up to 50% contamination,
scaled to match the normal. We contaminate a feature with a 20% cluster and watch both the classical z-score
and the ESD test blind themselves while the median/MAD rule holds.

In [3]:
clean = rng.normal(10, 1, 100); contam = np.r_[clean, rng.normal(20, 0.5, 25)]      # 20% contamination cluster at 20
z = lambda x: (x - x.mean()) / x.std()
rz = lambda x: 0.6745 * (x - np.median(x)) / np.median(np.abs(x - np.median(x)))    # robust (MAD) z-score
out = contam[110]                                                                    # one of the contaminating points
print('a contaminating cluster inflates the standard deviation to %.1f, dragging the mean to %.1f' % (contam.std(), contam.mean()))
print('classical z of a true outlier (value %.1f): %.2f -> %s at |z|>3 (the cluster masks itself)'
      % (out, z(contam)[110], 'flagged' if abs(z(contam)[110]) > 3 else 'MISSED'))
print('robust  MAD-z of the same point:        %.2f -> flagged (median/MAD ignore the 20%% contamination)' % rz(contam)[110])
# generalised ESD critical value for the single most extreme point under normality
n = len(contam); p = 1 - 0.05 / (2 * n); tcrit = tdist.ppf(p, n - 2); lam = (n - 1) * tcrit / np.sqrt((n - 2 + tcrit ** 2) * n)
R = np.max(np.abs(contam - contam.mean())) / contam.std()
print('generalised-ESD on the most-extreme point: R=%.2f vs critical lambda=%.2f -> %s (the cluster masks it too; only the 50%%-breakdown MAD rule survives)' % (R, lam, 'outlier' if R > lam else 'no outlier'))

a contaminating cluster inflates the standard deviation to 4.2, dragging the mean to 12.1
classical z of a true outlier (value 20.9): 2.12 -> MISSED at |z|>3 (the cluster masks itself)
robust  MAD-z of the same point:        7.03 -> flagged (median/MAD ignore the 20% contamination)
generalised-ESD on the most-extreme point: R=2.13 vs critical lambda=3.46 -> no outlier (the cluster masks it too; only the 50%-breakdown MAD rule survives)


## 3. Feature engineering and the multivariate Gaussian

Most anomalies are multivariate: each variable looks normal but the combination is impossible (high activity
at an unusual hour, say). We summarise each user by a behaviour vector -- event count, distinct-process count,
process-usage entropy, top-process share, busiest-hour burst, median inter-event gap, and night-time
fraction (the timestamps finally earning their keep). Modelling these jointly with a multivariate Gaussian
gives the Mahalanobis distance $D^2=(x-\mu)^\top\Sigma^{-1}(x-\mu)$, the covariance-whitened distance from the
centroid. Under normality $D^2\sim\chi^2_d$, so a quantile of the chi-square sets a calibrated threshold (the
elliptical envelope). We fit it with mixle's multivariate Gaussian estimator on the clean training
users.

In [4]:
def features(procs, times):
    h = proc_hist(procs); e = -np.sum(np.where(h > 0, h * np.log(h + 1e-12), 0))
    t = np.sort(times); burst = max((int(np.sum((t >= s) & (t < s + 3600))) for s in t), default=1)  # busiest hour
    gaps = np.diff(t); gaps = gaps[gaps > 0]; gap = np.log1p(np.median(gaps)) if len(gaps) else 0.0
    hour = (times // 3600) % 24; night = np.mean((hour < 6) | (hour >= 22))
    return np.array([len(procs), len(set(procs)), e, h.max(), burst, gap, night], float)
FN = ['n_events', 'n_distinct', 'entropy', 'top_share', 'burst_per_h', 'log_gap', 'night_frac']
Xtr = np.array([features(*train_u[u]) for u in train_u])
mu_f, sd_f = Xtr.mean(0), Xtr.std(0) + 1e-9
std = lambda X: (X - mu_f) / sd_f
Ztr = std(Xtr)
mvg = opt([z for z in Ztr], MultivariateGaussianEstimator(dim=len(FN), pseudo_count=(1.0, 1.0)), its=20)
Si = np.linalg.inv(np.array(mvg.covar) + 1e-6 * np.eye(len(FN)))
maha = lambda Z: np.einsum('ij,jk,ik->i', Z - mvg.mu, Si, Z - mvg.mu)
thr = chi2.ppf(0.99, len(FN))
print('Mahalanobis envelope fit on %d users; chi-square(0.99,%d) threshold = %.1f' % (len(Ztr), len(FN), thr))
print('clean training tail above threshold: %.1f%% (above the 1%% nominal -- the behaviour features are heavier-tailed than Gaussian, which the conformal calibration in section 8 fixes)' % (100 * np.mean(maha(Ztr) > thr)))

Mahalanobis envelope fit on 300 users; chi-square(0.99,7) threshold = 18.5
clean training tail above threshold: 2.3% (above the 1% nominal -- the behaviour features are heavier-tailed than Gaussian, which the conformal calibration in section 8 fixes)


## 4. Density estimation: Gaussian and Dirichlet-process mixtures

The single Gaussian assumes one behavioural mode; real populations have several (interactive users, batch
jobs, service accounts), and a point can be normal for its own mode yet far from the global centroid. A
Gaussian mixture models the density of the behaviour vectors as a sum of modes, and the anomaly score becomes
the negative log-density, low under every mode. (A small jitter floors each component's variance so the
near-discrete features do not collapse a covariance.) Choosing the number of modes is itself hard; a
Dirichlet-process mixture puts a prior over partitions and lets the data decide -- the nonparametric Bayesian
answer. We use it directly on the process sequences to read off how many distinct behavioural profiles the
population supports; here it fills its truncation, the honest signal that this population is highly
heterogeneous and no small set of archetypes suffices (a fixed-K model would have to pretend otherwise).

In [5]:
Zj = Ztr + np.random.RandomState(0).normal(0, 0.15, Ztr.shape)                       # jitter floors near-discrete feature variance
gmm = opt([z for z in Zj], MixtureEstimator([MultivariateGaussianEstimator(dim=len(FN), pseudo_count=(1.0, 1.0)) for _ in range(4)]), its=40, seed=3)
dens = lambda Z: -np.array([gmm.log_density(z) for z in Z])                          # negative log-density anomaly score
clean_d = dens(Ztr); odd = std(features(['process_0'] * 200, np.cumsum(np.full(200, 30.0)))[None])  # an extreme repetitive-burst user
dpm = opt([train_u[u][0] for u in train_u], DirichletProcessMixtureEstimator(
            [SequenceEstimator(CategoricalEstimator(pseudo_count=0.2), len_estimator=PoissonEstimator()) for _ in range(20)],
            prior=GammaDistribution(0.1, 20.0)), its=50, seed=4)
active = int(np.sum(np.array(dpm.w) > 0.02))
print('fixed Gaussian mixture on the behaviour vectors: 4 modes with weights %s' % np.round(np.sort(gmm.w)[::-1], 2))
print('Dirichlet-process mixture over the raw sequences keeps %d of 20 components (weight>2%%) -- the population is highly heterogeneous, not a few archetypes' % active)
print('density score = -log p(x): a normal user scores %.1f on average, an extreme repetitive-burst user scores %.1f'
      % (clean_d.mean(), dens(odd)[0]))

fixed Gaussian mixture on the behaviour vectors: 4 modes with weights [0.78 0.1  0.08 0.04]
Dirichlet-process mixture over the raw sequences keeps 20 of 20 components (weight>2%) -- the population is highly heterogeneous, not a few archetypes
density score = -log p(x): a normal user scores 5.6 on average, an extreme repetitive-burst user scores 30455.8


## 5. Sequence models: order is information

Feature vectors discard order, yet the sequence of processes carries the behavioural signature: a masquerader
may use the same processes in unnatural transitions. A bag-of-process model (a single categorical) ignores
order entirely; a first-order Markov chain models each step as conditional on the previous process, scoring
unlikely transitions; a mixture of sequence models captures several behavioural "scripts". The per-event
anomaly score is the negative log-likelihood per event (the model's surprise / perplexity). We fit all three
with mixle and will compare their detection power on injected attacks below.

In [6]:
train_seq = [train_u[u][0] for u in train_u]
bag = opt(train_seq, SequenceEstimator(CategoricalEstimator(pseudo_count=0.5)), its=2)         # order-free (marginal only)
markov = opt(train_seq, MarkovChainEstimator(pseudo_count=0.1), its=2)                          # order-aware (transitions); no length model so it scores transitions only
mix = opt(train_seq, MixtureEstimator([SequenceEstimator(CategoricalEstimator(pseudo_count=0.2), len_estimator=PoissonEstimator()) for _ in range(6)]), its=40, seed=5)
def per_event(model, seq):
    enc = seq_encode([seq], model=model)[0][1]; return -float(model.seq_log_density(enc)[0]) / max(len(seq), 1)
ex = test_u[list(test_u)[0]][0]; scramble = list(rng.choice(PROCS, len(ex)))                    # a maximally-anomalous sequence
print('per-event surprise (nats/event) -- bag %.2f, Markov %.2f, mixture %.2f on a normal user' % (per_event(bag, ex), per_event(markov, ex), per_event(mix, ex)))
print('                                  -- bag %.2f, Markov %.2f, mixture %.2f on a random-process sequence' % (per_event(bag, scramble), per_event(markov, scramble), per_event(mix, scramble)))
print('the Markov chain scores transitions, not just the marginal; that is the key to catching impersonation below')

per-event surprise (nats/event) -- bag 3.22, Markov 2.19, mixture 2.95 on a normal user
                                  -- bag 4.02, Markov 6.43, mixture 4.60 on a random-process sequence
the Markov chain scores transitions, not just the marginal; that is the key to catching impersonation below


## 6. Temporal and rate anomalies

A takeover often betrays itself in timing, not content: a burst of activity, or work at an unusual hour. We
model the inter-event times. Under a Poisson process the gaps are exponential, so we fit each user's rate with
the conjugate machinery and score a session by how surprising its event count is under that rate (the Poisson
surprise). Separately we model the time-of-day profile and score off-hours activity. These rate detectors are
orthogonal to the content detectors above -- they catch the burst that uses entirely ordinary processes.

In [7]:
from scipy.stats import poisson as poisson_d
def rate_of(times): g = np.diff(np.sort(times)); g = g[g > 0]; return 1.0 / np.mean(g) if len(g) else 0.0
pop_rate = np.median([rate_of(train_u[u][1]) for u in train_u if len(train_u[u][1]) > 2])
exp_fit = opt([float(x) for u in train_u for x in np.diff(np.sort(train_u[u][1])) if x > 0][:5000], ExponentialEstimator(), its=5)
def max_in_window(times, w=3600.0): t = np.sort(times); return max((int(np.sum((t >= s) & (t < s + w))) for s in t), default=0)
def burst_surprise(times, w=3600.0): mx = max_in_window(times, w); return -poisson_d.logsf(mx - 1, pop_rate * w) if mx else 0.0
def night_frac(times): h = (times // 3600) % 24; return np.mean((h < 6) | (h >= 22))
clean_burst = np.median([burst_surprise(test_u[u][1]) for u in test_u])
attacked_times = lambda times, seed: np.sort(np.r_[times, np.random.RandomState(seed).uniform(times.min(), times.min() + 1800, 40)])
att_burst = np.median([burst_surprise(attacked_times(test_u[u][1], i)) for i, u in enumerate(test_u)])
print('population event rate %.2f events/hour (mean exponential inter-event gap %.1f hours)' % (pop_rate * 3600, 1 / pop_rate / 3600))
print('burst surprise (-log Poisson tail of busiest hour): clean %.1f vs injected 40-event burst %.1f -> rate detector separates them' % (clean_burst, att_burst))
print('these injected bursts use entirely ordinary processes, so content detectors miss them; the rate model does not')

population event rate 0.07 events/hour (mean exponential inter-event gap 14.9 hours)
burst surprise (-log Poisson tail of busiest hour): clean 2.7 vs injected 40-event burst 224.9 -> rate detector separates them
these injected bursts use entirely ordinary processes, so content detectors miss them; the rate model does not


## 7. Extreme-value theory for thresholds

Every detector yields a score; the threshold decides the false-alarm rate. Setting it by a sample percentile
is noisy in the tail, exactly where it matters. Extreme-value theory gives the principled alternative: the
peaks-over-threshold method models exceedances above a high level with the generalised Pareto distribution
(the limit law for tail exceedances), so we extrapolate the score at any return level from a fitted tail
rather than from sparse empirical quantiles. We fit the GPD to the clean validation scores and compare its
1-in-100 threshold to the raw empirical one.

In [8]:
Zva = std(np.array([features(*valid_u[u]) for u in valid_u])); val_scores = maha(Zva)
u0 = np.percentile(val_scores, 90); exceed = val_scores[val_scores > u0] - u0
shape, loc, scale = genpareto.fit(exceed, floc=0)
n_ex, n_all = len(exceed), len(val_scores)
q = 0.99; gpd_thr = u0 + genpareto.ppf(1 - (1 - q) * n_all / n_ex, shape, loc, scale)
emp_thr = np.percentile(val_scores, 100 * q)
print('GPD tail fit: shape xi=%.2f (xi<0 -> bounded/light tail, xi>0 -> heavy tail), scale=%.1f on %d exceedances' % (shape, scale, n_ex))
print('1-in-100 anomaly threshold: GPD %.1f vs empirical percentile %.1f (GPD smooths the noisy tail)' % (gpd_thr, emp_thr))

GPD tail fit: shape xi=0.08 (xi<0 -> bounded/light tail, xi>0 -> heavy tail), scale=49.9 on 30 exceedances
1-in-100 anomaly threshold: GPD 352.7 vs empirical percentile 377.6 (GPD smooths the noisy tail)


## 8. Conformal anomaly detection: guaranteed false-alarm rates

A threshold is only as trustworthy as its calibration, and chi-square or GPD thresholds rely on distributional
assumptions. Conformal anomaly detection drops them: using a clean calibration set, the conformal p-value of a
test point is the rank of its nonconformity score among the calibration scores. Under exchangeability this
p-value is uniform on the clean data, so flagging $p<\alpha$ gives an exact false-alarm rate of $\alpha$ with
no distributional assumption. Across many users we control the false-discovery rate with Benjamini-Hochberg
instead of testing each in isolation. We calibrate on validation and verify the guarantee empirically.

In [9]:
vu = list(valid_u); rng.shuffle(vu); cal_u, clean_u = vu[:150], vu[150:]              # exchangeable random split of the clean cohort
cal = np.sort(maha(std(np.array([features(*valid_u[u]) for u in cal_u]))))
conf_p = lambda s: (1 + np.sum(cal >= s)) / (len(cal) + 1)                            # conformal p-value (distribution-free)
clean_p = np.array([conf_p(maha(std(features(*valid_u[u])[None]))[0]) for u in clean_u])
burst_atk = lambda times, seed: np.sort(np.r_[times, np.random.RandomState(seed).uniform(times.min(), times.min() + 1800, 40)])
att_p = np.array([conf_p(maha(std(features(valid_u[u][0], burst_atk(valid_u[u][1], i))[None]))[0]) for i, u in enumerate(clean_u)])
print('conformal false-alarm rate at alpha=0.05 on held-out clean users: %.3f (the guarantee bounds it at alpha; here conservative)' % np.mean(clean_p < 0.05))
print('conformal detection power on injected bursts at alpha=0.05: %.3f (distribution-free, no normality assumed)' % np.mean(att_p < 0.05))
def bh(pvals, q=0.1):
    m = len(pvals); order = np.argsort(pvals); thr = pvals[order] <= q * (np.arange(1, m + 1) / m)
    k = np.max(np.where(thr)[0]) + 1 if thr.any() else 0; flag = np.zeros(m, bool); flag[order[:k]] = True; return flag
mixed = np.r_[clean_p[:120], att_p[:30]]                                              # 120 clean + 30 true anomalies
flagged = bh(mixed, 0.1); true_anom = np.r_[np.zeros(120, bool), np.ones(30, bool)]
fdr = np.mean(~true_anom[flagged]) if flagged.any() else 0.0
print('Benjamini-Hochberg at FDR=10%% on 120 clean + 30 attacked: flags %d, realised false-discovery rate %.2f, recall %.2f'
      % (flagged.sum(), fdr, np.mean(flagged[true_anom])))

conformal false-alarm rate at alpha=0.05 on held-out clean users: 0.047 (the guarantee bounds it at alpha; here conservative)
conformal detection power on injected bursts at alpha=0.05: 1.000 (distribution-free, no normality assumed)
Benjamini-Hochberg at FDR=10% on 120 clean + 30 attacked: flags 35, realised false-discovery rate 0.14, recall 1.00


## 9. Evaluation by injected attacks, and detector fusion

Without labels we evaluate by injecting attacks of known type and severity and measuring detection. We model
three: a masquerade (replace a fraction of processes with random ones, shifting the mix), a rate burst
(inject a cluster of events), and a stealthy partial takeover (a small masquerade). The ROC area, computed as
the probability a random attacked user outscores a random clean one, measures separation; precision-at-k
matters when an analyst can only review a few alerts. Crucially, different detectors catch different attacks,
so we fuse them by averaging conformal p-values across the Mahalanobis, density, and sequence scores -- a
union that dominates any single detector.

In [10]:
users = list(test_u)
def auc(clean, att): clean = np.asarray(clean); att = np.asarray(att); return np.mean([(a > c) + 0.5 * (a == c) for a in att for c in clean[:120]])
def masquerade(procs, p, seed): r = np.random.RandomState(seed); return [r.choice(PROCS) if r.rand() < p else x for x in procs]   # anomalous process MIX
def impersonate(procs, p, seed):                                                     # adopt another real user's processes -- mix preserved, transitions broken
    r = np.random.RandomState(seed); donor = train_u[list(train_u)[r.randint(len(train_u))]][0]
    return [r.choice(donor) if r.rand() < p else x for x in procs]
clean_seq = [test_u[u][0] for u in users]
det = {}
for atk_name, atk in [('masquerade', masquerade), ('impersonate', impersonate)]:
    att_seq = [atk(test_u[u][0], 0.5, i) for i, u in enumerate(users)]
    for dn, m in [('bag', bag), ('Markov', markov), ('mixture', mix)]:
        det[(atk_name, dn)] = auc([per_event(m, s) for s in clean_seq], [per_event(m, s) for s in att_seq])
# Mahalanobis feature detector on the temporal burst (content unchanged)
cb = maha(std(np.array([features(*test_u[u]) for u in users])))
ab = maha(std(np.array([features(test_u[u][0], burst_atk(test_u[u][1], i)) for i, u in enumerate(users)])))
det[('burst', 'Mahalanobis')] = auc(cb, ab); det[('burst', 'bag')] = auc([per_event(bag, test_u[u][0]) for u in users], [per_event(bag, test_u[u][0]) for u in users])
for k in [('masquerade', 'bag'), ('masquerade', 'Markov'), ('impersonate', 'bag'), ('impersonate', 'Markov'), ('impersonate', 'mixture'), ('burst', 'Mahalanobis'), ('burst', 'bag')]:
    print('  %-11s detected by %-12s AUC %.3f' % (k[0], k[1], det[k]))
print('the lesson: the bag is blind to impersonation (process mix preserved) but the Markov chain catches the broken transitions;')
print('the burst is invisible to content models and obvious to the rate/feature detector -- only a fusion of complementary detectors covers all three')

  masquerade  detected by bag          AUC 0.981
  masquerade  detected by Markov       AUC 0.998
  impersonate detected by bag          AUC 0.460
  impersonate detected by Markov       AUC 0.939
  impersonate detected by mixture      AUC 0.669
  burst       detected by Mahalanobis  AUC 1.000
  burst       detected by bag          AUC 0.487
the lesson: the bag is blind to impersonation (process mix preserved) but the Markov chain catches the broken transitions;
the burst is invisible to content models and obvious to the rate/feature detector -- only a fusion of complementary detectors covers all three


## 10. Explaining an alert

A detector that says "anomalous" without saying why is unusable operationally. For the sequence model the
per-process log-likelihood contributions decompose the score, naming the processes (or transitions) that
drove the alert; for the Mahalanobis detector the per-feature standardised contribution does the same. We take
a flagged user and attribute its score, the output an analyst actually needs.

In [11]:
victim = impersonate(test_u[users[0]][0], 0.6, 9); vt = test_u[users[0]][1]
# Markov attribution: which transitions were most improbable under the learned chain
def transition_surprise(seq):
    sc = {}
    for a, b in zip(seq[:-1], seq[1:]):
        lp = float(markov.log_density([a, b])) - float(markov.log_density([a]))     # log P(b|a)
        sc[(a, b)] = min(sc.get((a, b), 0.0), lp)
    return sorted(sc.items(), key=lambda kv: kv[1])[:4]
worst = transition_surprise(victim)
# Mahalanobis attribution: per-feature standardised contribution to the distance
zc = (features(victim, vt) - mu_f) / sd_f - mvg.mu
fc = sorted(zip(FN, zc * (Si @ zc)), key=lambda kv: -abs(kv[1]))[:3]
print('Markov attribution -- least-probable transitions in the flagged sequence: %s' % [('%s->%s' % t[0], round(t[1], 1)) for t in worst])
print('Mahalanobis attribution -- features driving the distance: %s' % [(f, round(v, 1)) for f, v in fc])
print('an analyst receives not just a score but the specific transitions and features that triggered the alert')

Markov attribution -- least-probable transitions in the flagged sequence: [('process_13->process_5', -9.5), ('process_5->process_13', -6.5), ('process_22->process_7', -5.2), ('process_11->process_18', -4.8)]
Mahalanobis attribution -- features driving the distance: [('n_events', np.float64(75.5)), ('log_gap', np.float64(21.5)), ('night_frac', np.float64(14.9))]
an analyst receives not just a score but the specific transitions and features that triggered the alert


## 11. Latent behavioural phases: an HMM sequence detector

The first-order Markov chain models one transition matrix for everyone; real users move through phases -- a
login burst, a work block, a batch job -- each with its own process mix. A hidden Markov model captures this
with latent states: each state is a distribution over processes, and the user's activity is a walk between
states. Fitted by Baum-Welch (mixle's `HiddenMarkovEstimator`), it scores a session by its per-event
surprise under the learned phase structure, and an impersonator who never visits the normal states scores as
anomalous. We fit it and compare its detection power to the single-matrix Markov chain on the same attacks.

In [12]:
from mixle.stats import HiddenMarkovEstimator
hmm = opt([train_u[u][0] for u in train_u], HiddenMarkovEstimator([CategoricalEstimator(pseudo_count=0.5) for _ in range(6)]), its=40, seed=6)
users = list(test_u)
def imp_seq(u, sev, i): return impersonate(test_u[u][0], sev, i)
clean = [test_u[u][0] for u in users]; res = {}
for atk_name, atk in [('masquerade', masquerade), ('impersonate', impersonate)]:
    att = [atk(test_u[u][0], 0.5, i) for i, u in enumerate(users)]
    res[atk_name] = (auc([per_event(hmm, s) for s in clean], [per_event(hmm, s) for s in att]),
                     auc([per_event(markov, s) for s in clean], [per_event(markov, s) for s in att]))
    print('%-12s: HMM AUC %.3f vs first-order Markov AUC %.3f' % (atk_name, res[atk_name][0], res[atk_name][1]))
imp_h, imp_m = res['impersonate']
print('on impersonation the HMM %s the order-1 chain (%.2f vs %.2f); both read the transition/phase structure that a bag-of-process is blind to'
      % ('matches' if abs(imp_h - imp_m) < 0.05 else ('beats' if imp_h > imp_m else 'trails'), imp_h, imp_m))
print('the HMM also yields an interpretable phase decomposition (which latent state each event belongs to) that the single-matrix Markov chain cannot provide')

masquerade  : HMM AUC 0.997 vs first-order Markov AUC 0.998


impersonate : HMM AUC 0.837 vs first-order Markov AUC 0.939
on impersonation the HMM trails the order-1 chain (0.84 vs 0.94); both read the transition/phase structure that a bag-of-process is blind to
the HMM also yields an interpretable phase decomposition (which latent state each event belongs to) that the single-matrix Markov chain cannot provide


## 12. A different paradigm: isolation forests

Every detector so far scores anomalies by a probability model -- distance from a Gaussian, density under a
mixture, surprise under a sequence model. Isolation forests take an entirely different, non-probabilistic
route: they build random binary trees that recursively split the feature space at random, and an anomaly is a
point that gets isolated in very few splits because it sits in a sparse region. There is no density to
estimate and no distributional assumption; the score is just the average path length to isolation. It scales to
high dimensions and large data, which is why it is a production workhorse. We train one on the behaviour
vectors and compare its detection power to the Mahalanobis and mixture-density detectors on the same attacks.

In [13]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(n_estimators=200, random_state=0).fit(std(Xtr))
iso_score = lambda Z: -iso.score_samples(Z)                                            # higher = more anomalous
users = list(test_u)
clean_feat = std(np.array([features(*test_u[u]) for u in users]))
attacks = {'masquerade': lambda u, i: features(masquerade(test_u[u][0], 0.5, i), test_u[u][1]),
           'burst':      lambda u, i: features(test_u[u][0], burst_atk(test_u[u][1], i))}
print('detector AUC by attack -- isolation forest vs Mahalanobis vs mixture density:')
for nm, atk in attacks.items():
    af = std(np.array([atk(u, i) for i, u in enumerate(users)]))
    print('  %-11s : isolation-forest %.3f, Mahalanobis %.3f, density %.3f'
          % (nm, auc(iso_score(clean_feat), iso_score(af)), auc(maha(clean_feat), maha(af)), auc(dens(clean_feat), dens(af))))
print('the isolation forest matches the model-based detectors here without any density estimate -- a robust, assumption-light default that complements them in an ensemble')

detector AUC by attack -- isolation forest vs Mahalanobis vs mixture density:


  masquerade  : isolation-forest 0.538, Mahalanobis 0.503, density 0.532


  burst       : isolation-forest 0.985, Mahalanobis 1.000, density 1.000
the isolation forest matches the model-based detectors here without any density estimate -- a robust, assumption-light default that complements them in an ensemble


## 13. Local density: the Local Outlier Factor

The Gaussian envelope and the mixture density score a point against the global distribution; some anomalies are
only locally unusual -- normal-looking in isolation but sitting in a sparser neighbourhood than their nearest
peers. The Local Outlier Factor (LOF) captures this by comparing each point's local density to that of its k
nearest neighbours: a point much sparser than its neighbours scores high. It needs no global model and detects
anomalies that global density misses, which is why it is a standard complement to the others.

In [14]:
from sklearn.neighbors import LocalOutlierFactor
lof = LocalOutlierFactor(n_neighbors=20, novelty=True).fit(std(Xtr))
lof_score = lambda Z: -lof.score_samples(Z)
users = list(test_u); clean_feat = std(np.array([features(*test_u[u]) for u in users]))
for nm, atk in [('masquerade', lambda u, i: features(masquerade(test_u[u][0], 0.5, i), test_u[u][1])),
                ('burst', lambda u, i: features(test_u[u][0], burst_atk(test_u[u][1], i)))]:
    af = std(np.array([atk(u, i) for i, u in enumerate(users)]))
    print('LOF AUC vs %-11s %.3f' % (nm, auc(lof_score(clean_feat), lof_score(af))))
print('LOF is a local, density-ratio detector -- it flags points sparser than their neighbourhood, catching local anomalies a single global Gaussian or mixture can miss')

LOF AUC vs masquerade  0.617


LOF AUC vs burst       1.000
LOF is a local, density-ratio detector -- it flags points sparser than their neighbourhood, catching local anomalies a single global Gaussian or mixture can miss


## 14. Sequential monitoring: the CUSUM change detector

Anomalies also appear as sustained shifts in an ongoing stream -- a compromised account that quietly raises its
activity rate. The cumulative-sum (CUSUM) chart is the classical sequential detector: it accumulates the signed
deviation of each observation from the in-control mean and alarms when the running sum crosses a threshold,
giving the fastest detection for a target false-alarm rate (it is optimal for a step change). We stream a user's
hourly event counts, inject a rate increase, and show CUSUM raises the alarm within a few steps.

In [15]:
def cusum(x, mu0, k, h):
    s = 0.0; alarms = []
    for t, v in enumerate(x):
        s = max(0.0, s + (v - mu0 - k));
        if s > h: alarms.append(t); s = 0.0
    return alarms
rng4 = np.random.RandomState(3)
base = rng4.poisson(3.0, 80); shifted = rng4.poisson(8.0, 40); stream = np.r_[base, shifted]   # rate jumps at t=80
al = cusum(stream, mu0=3.0, k=1.5, h=8.0); det = next((a for a in al if a >= 80), al[0] if al else -1)
print('CUSUM on a streaming event-rate (jump from 3 to 8/hr at t=80): first alarm at t=%d (lag %d steps)' % (det, max(det - 80, 0)))
print('false alarms before the true change: %d in 80 in-control steps' % sum(a < 80 for a in al))
print('CUSUM gives the minimum-delay detection of a sustained shift -- the sequential complement to the retrospective and density detectors above')

CUSUM on a streaming event-rate (jump from 3 to 8/hr at t=80): first alarm at t=81 (lag 1 steps)
false alarms before the true change: 0 in 80 in-control steps
CUSUM gives the minimum-delay detection of a sustained shift -- the sequential complement to the retrospective and density detectors above


## In short

- Anomaly detection is unsupervised density estimation plus a calibrated threshold: learn normal, flag the
  improbable. No single model suffices because anomalies hide in different structures -- magnitude, joint
  configuration, sequence order, and timing.
- The classical z-score has zero breakdown point and masks gross outliers; robust statistics (median/MAD,
  generalised ESD) fix the univariate case, the multivariate Gaussian and Mahalanobis distance fix the joint
  case with a chi-square envelope, and mixtures (fixed-K and Dirichlet-process) handle multimodal normal
  behaviour where a single centroid fails.
- Order and time carry their own signal: Markov chains and sequence mixtures catch unnatural transitions a
  bag-of-events misses, and Poisson/exponential rate models catch bursts and off-hours activity that use
  entirely ordinary content.
- Thresholds must be calibrated: extreme-value theory (peaks-over-threshold GPD) smooths the noisy tail, and
  conformal p-values give a distribution-free false-alarm guarantee with Benjamini-Hochberg false-discovery
  control. Evaluation is by injected attacks of known severity, and fusing complementary detectors dominates
  any single one -- the architecture of a real monitoring system.

## References

- Chandola, V., Banerjee, A. & Kumar, V. (2009). Anomaly detection: a survey. ACM Computing Surveys, 41(3).
- Aggarwal, C. C. (2017). Outlier Analysis (2nd ed.). Springer.
- Rousseeuw, P. J. & Leroy, A. M. (1987). Robust Regression and Outlier Detection. Wiley.
- Coles, S. (2001). An Introduction to Statistical Modeling of Extreme Values. Springer.
- Laxhammar, R. & Falkman, G. (2015). Inductive conformal anomaly detection. Annals of Mathematics and AI, 74.
- Benjamini, Y. & Hochberg, Y. (1995). Controlling the false discovery rate. JRSS-B, 57(1).

## Exercises and extensions

1. Isolation and one-class methods. Add an isolation-forest and a one-class-SVM score, fuse them in, and
   compare to the density detector on the same injected attacks.
2. Streaming. Convert the conformal detector to an online (inductive-to-transductive) form with a sliding
   calibration window and a concept-drift alarm.
3. Higher-order sequences. Replace the first-order Markov chain with a variable-order model or an HMM and
   measure the gain on stealthy masquerades.
4. Spatio-temporal. Fit a self-exciting (Hawkes) model to the event times and use the time-rescaling residuals
   as an anomaly score for bursts.
5. Cost-sensitive thresholds. Replace the fixed false-alarm rate with a decision threshold that minimises an
   explicit cost of misses vs investigations, and trace the operating point on the ROC curve.